# Stress Testing

## Goal

Evaluate how expected credit loss (ECL) changes under worsening credit conditions.

### Scenarios

- **Base Case:** Historical PD
- **Moderate Stress:** Base PD × 1.25
- **Severe Stress:** Base PD × 1.50

These stress scenarios are assumptions used to evaluate portfolio sensitivity, not forecasts.

In [1]:
import pandas as pd

df = pd.read_csv("../data/credit_risk_ecl.csv")

df.head()

,loan_id,risk_score,risk_tier,default_flag,PD,LGD,EAD,ECL,purpose,annual_inc,income_segment
0,68407277,30,Medium,0,0.174541,0.45,3600.0,282.756452,debt_consolidation,55000.0,$40K-$60K
1,68355089,15,Low,0,0.111075,0.45,24700.0,1234.599836,small_business,65000.0,$60K-$80K
2,68341763,40,Medium,0,0.174541,0.45,20000.0,1570.869176,home_improvement,63000.0,$60K-$80K
3,68476807,65,Very High,0,0.286666,0.45,10400.0,1341.598081,major_purchase,104433.0,$100K+
4,68426831,50,High,0,0.226972,0.45,11950.0,1220.541094,debt_consolidation,34000.0,< $40K


## Stress Scenario PDs

### Moderate Stress

Increase the base PD by 25%:

**Stress PD = Base PD × 1.25**

### Severe Stress

Increase the base PD by 50%:

**Stress PD = Base PD × 1.50**

PD is capped at 1.0 because probability of default cannot exceed 100%.

In [3]:
df["PD_moderate"] = (df["PD"] * 1.25).clip(upper=1.0)
df["PD_severe"] = (df["PD"] * 1.50).clip(upper=1.0)

df[["PD", "PD_moderate", "PD_severe"]].head()

,PD,PD_moderate,PD_severe
0,0.174541,0.218176,0.261812
1,0.111075,0.138844,0.166613
2,0.174541,0.218176,0.261812
3,0.286666,0.358333,0.429999
4,0.226972,0.283715,0.340458


## Stress ECL

Recalculate expected credit loss under each stress scenario:

**Stress ECL = Stress PD × LGD × EAD**

LGD and EAD remain unchanged so the analysis isolates the effect of worsening default risk.

In [5]:
df["ECL_moderate"] = df["PD_moderate"] * df["LGD"] * df["EAD"]
df["ECL_severe"] = df["PD_severe"] * df["LGD"] * df["EAD"]

df[["ECL", "ECL_moderate", "ECL_severe"]].head()

,ECL,ECL_moderate,ECL_severe
0,282.756452,353.445564,424.134677
1,1234.599836,1543.249795,1851.899754
2,1570.869176,1963.586469,2356.303763
3,1341.598081,1676.997601,2012.397121
4,1220.541094,1525.676368,1830.811641


## Portfolio Stress Test Results

Compare total expected credit loss across the Base, Moderate, and Severe stress scenarios.

In [6]:
base_ecl = df["ECL"].sum()
moderate_ecl = df["ECL_moderate"].sum()
severe_ecl = df["ECL_severe"].sum()

scenario_summary = pd.DataFrame({
    "Scenario": ["Base", "Moderate", "Severe"],
    "Total_ECL": [base_ecl, moderate_ecl, severe_ecl]
})

scenario_summary

,Scenario,Total_ECL
0,Base,1.748556e+09
1,Moderate,2.185695e+09
2,Severe,2.622834e+09


## ECL Increase %

Measure how much total expected credit loss increases relative to the Base Case.

**ECL Increase % = ((Stress ECL - Base ECL) / Base ECL) × 100**

In [7]:
moderate_increase_pct = ((moderate_ecl - base_ecl) / base_ecl) * 100
severe_increase_pct = ((severe_ecl - base_ecl) / base_ecl) * 100

print(f"Moderate Stress ECL Increase: {moderate_increase_pct:.2f}%")
print(f"Severe Stress ECL Increase: {severe_increase_pct:.2f}%")

Moderate Stress ECL Increase: 25.00%
Severe Stress ECL Increase: 50.00%


## Business Question 1: Which borrowers create the largest potential losses?

Identify the loans with the highest expected credit loss under the Severe Stress scenario.

In [8]:
top_loss_borrowers = df.nlargest(10, "ECL_severe")[
    ["loan_id", "risk_tier", "PD_severe", "LGD", "EAD", "ECL_severe"]
]

top_loss_borrowers

,loan_id,risk_tier,PD_severe,LGD,EAD,ECL_severe
375907,130799512,Very High,0.429999,0.45,40000.0,7739.988928
375980,130890615,Very High,0.429999,0.45,40000.0,7739.988928
376483,130653045,Very High,0.429999,0.45,40000.0,7739.988928
377238,130464147,Very High,0.429999,0.45,40000.0,7739.988928
377641,130331377,Very High,0.429999,0.45,40000.0,7739.988928
378660,130289212,Very High,0.429999,0.45,40000.0,7739.988928
378730,130250263,Very High,0.429999,0.45,40000.0,7739.988928
378943,130184483,Very High,0.429999,0.45,40000.0,7739.988928
379434,129882303,Very High,0.429999,0.45,40000.0,7739.988928
380463,129824888,Very High,0.429999,0.45,40000.0,7739.988928


### Finding

The largest potential losses under the Severe Stress scenario come from **Very High risk borrowers with large exposures**. The top loans have an EAD of $40,000 and a stressed PD of approximately 43%, resulting in an ECL of about $7,740 per loan.

This shows that potential credit losses are driven by the combination of **high default risk and high exposure**.

## Business Question 2: Which loan purposes are most vulnerable?

Compare total expected credit loss by loan purpose under the Severe Stress scenario.

In [9]:
purpose_stress = (
    df.groupby("purpose")["ECL_severe"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

purpose_stress

,purpose,ECL_severe
0,debt_consolidation,1.634933e+09
1,credit_card,6.070149e+08
2,home_improvement,1.486308e+08
3,other,9.661776e+07
4,major_purchase,3.998396e+07
5,small_business,2.895241e+07
6,medical,1.768880e+07
7,car,1.530965e+07
8,house,1.281729e+07
9,moving,9.509476e+06


### Finding

**Debt consolidation** loans contribute the largest total expected credit loss under the Severe Stress scenario, followed by **credit card** and **home improvement** loans.

This indicates that potential stressed losses are heavily concentrated in the debt consolidation segment of the portfolio.

## Business Question 3: Which risk tiers contribute most to ECL?

Compare total expected credit loss across risk tiers under the Severe Stress scenario.

In [10]:
risk_tier_stress = (
    df.groupby("risk_tier")["ECL_severe"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

risk_tier_stress

,risk_tier,ECL_severe
0,High,9.647493e+08
1,Medium,9.123156e+08
2,Very High,5.615381e+08
3,Low,1.842309e+08


### Finding

The **High risk tier** contributes the largest total expected credit loss under the Severe Stress scenario, followed closely by the **Medium risk tier**.

Although Very High risk borrowers have greater individual default risk, total portfolio loss depends on both **risk level and portfolio exposure**. This shows that the highest-risk borrowers do not necessarily contribute the largest total portfolio loss.

## Business Question 4: Where is the portfolio concentrated?

Compare total exposure (EAD) across risk tiers to identify where the portfolio's credit exposure is concentrated. Which risk tier contains the largest total dollar exposure?

In [11]:
exposure_by_tier = (
    df.groupby("risk_tier")["EAD"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

exposure_by_tier

,risk_tier,EAD
0,Medium,7.743616e+09
1,High,6.297072e+09
2,Very High,2.902010e+09
3,Low,2.457208e+09


### Finding

The portfolio's largest exposure is concentrated in the **Medium risk tier**, with approximately $7.74 billion in EAD, followed by the **High risk tier** with approximately $6.30 billion.

This helps explain why the Medium and High risk tiers contribute substantial total ECL even though Very High borrowers have greater individual default risk.

## Business Question 5: What happens to total ECL under stress?

The portfolio's expected credit loss increases as default risk rises under the stress scenarios.

- **Base Case ECL:** approximately $1.75 billion
- **Moderate Stress ECL:** approximately $2.19 billion
- **Severe Stress ECL:** approximately $2.62 billion
- **Moderate Stress Increase:** 25%
- **Severe Stress Increase:** 50%

### Finding

Under the Severe Stress scenario, total expected credit loss increases from approximately **$1.75 billion to $2.62 billion**, an increase of **50%**.

This demonstrates the portfolio's sensitivity to worsening default risk and shows how stress testing can help estimate potential losses under adverse credit conditions.

In [12]:
df.to_csv("../data/credit_risk_stress_test.csv", index=False)

print("Stress test results saved successfully.")

Stress test results saved successfully.
